In [1]:
# ================== FLUJO PRINCIPAL COMPLETO ==================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
from shapely.geometry import Point
from io import BytesIO
import os
from datetime import datetime
import requests
import openpyxl
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Font, PatternFill, Alignment

def cargar_datos(archivo):
    df = pd.read_excel(archivo, sheet_name=None)
    df_clientes = df[list(df.keys())[0]]
    df_cdr = df[list(df.keys())[1]]
    gdf_clientes = gpd.GeoDataFrame(df_clientes, geometry=gpd.points_from_xy(df_clientes['U_longitud'], df_clientes['U_latitud']), crs='EPSG:4326')
    gdf_cdr = gpd.GeoDataFrame(df_cdr, geometry=gpd.points_from_xy(df_cdr['U_longitud'], df_cdr['U_latitud']), crs='EPSG:4326')
    return gdf_clientes, gdf_cdr

def analizar_distancias_osrm(gdf_clientes, gdf_cdr, cantidad=4415, tolerancia=2):
    url_base = 'http://router.project-osrm.org/route/v1/driving/'
    resultados = []
    for idx, cliente in gdf_clientes.head(cantidad).iterrows():
        distancias = {}
        for _, cdr in gdf_cdr.iterrows():
            coords = f"{cliente['U_longitud']},{cliente['U_latitud']};{cdr['U_longitud']},{cdr['U_latitud']}"
            url = url_base + coords + '?overview=false'
            try:
                r = requests.get(url)
                data = r.json()
                km = data['routes'][0]['distance']/1000 if 'routes' in data and data['routes'] else None
            except:
                km = None
            distancias[cdr['CodMET']] = km
        mejor_opcion = min(distancias, key=distancias.get) if all(distancias.values()) else None
        ahorro_km = abs(distancias['MET Celaya'] - distancias['MET Querétaro']) if all(distancias.values()) else None
        neutral = abs(distancias['MET Celaya'] - distancias['MET Querétaro']) <= tolerancia if all(distancias.values()) else None
        mejor = 'Se queda igual' if neutral else mejor_opcion
        resultados.append({
            'CodCli': cliente.get('CodCli',''),
            'Nombre': cliente.get('Nombre',''),
            'Ruta': cliente.get('Ruta',''),
            'U_latitud': cliente['U_latitud'],
            'U_longitud': cliente['U_longitud'],
            'Dist_Celaya_km': distancias.get('MET Celaya', None),
            'Dist_Querétaro_km': distancias.get('MET Querétaro', None),
            'Mejor_Opcion': mejor,
            'Ahorro_km': ahorro_km
        })
    return pd.DataFrame(resultados)

def crear_visualizaciones(df_analisis):
    buf = BytesIO()
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    colores = ['#FF6B00', '#00529B', '#FFA500']
    asignacion = df_analisis['Mejor_Opcion'].value_counts()
    axs[0].pie(asignacion.values, labels=asignacion.index, autopct='%1.1f%%', colors=colores[:len(asignacion)], startangle=90)
    axs[0].set_title('Distribución de Asignación de Clientes', fontsize=13, color='#FF6B00')
    axs[1].hist(df_analisis['Dist_Celaya_km'], bins=25, alpha=0.7, label='MET Celaya', color='#FF6B00', edgecolor='black')
    axs[1].hist(df_analisis['Dist_Querétaro_km'], bins=25, alpha=0.7, label='MET Querétaro', color='#00529B', edgecolor='black')
    axs[1].set_xlabel('Distancia (km)', fontsize=11)
    axs[1].set_ylabel('Frecuencia', fontsize=11)
    axs[1].set_title('Histograma de Distancias por MEET', fontsize=13, color='#FF6B00')
    axs[1].legend(fontsize=11)
    axs[1].grid(True, alpha=0.3)
    colores_disp = {'MET Celaya': '#FF6B00', 'MET Querétaro': '#00529B', 'Se queda igual': '#FFA500'}
    for meet, color in colores_disp.items():
        subset = df_analisis[df_analisis['Mejor_Opcion'] == meet]
        axs[2].scatter(subset['U_longitud'], subset['U_latitud'], s=20, alpha=0.7, c=color, label=meet)
    axs[2].set_xlabel('Longitud', fontsize=11)
    axs[2].set_ylabel('Latitud', fontsize=11)
    axs[2].set_title('Dispersión de Clientes por MEET', fontsize=13, color='#FF6B00')
    axs[2].legend(fontsize=11)
    plt.tight_layout()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf

def exportar_excel(df_analisis, directorio, timestamp, cantidad, grafica_buf):
    archivo_excel = os.path.join(directorio, f'Analisis_CDR_Adaptable_{len(df_analisis)}_Clientes_{timestamp}.xlsx')
    with pd.ExcelWriter(archivo_excel, engine='openpyxl') as writer:
        resumen = [
            ['📊 MÉTRICAS PRINCIPALES', ''],
            ['👥 Total clientes analizados', len(df_analisis)],
            ['🏢 MET Celaya', len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya'])],
            ['🏢 MET Querétaro', len(df_analisis[df_analisis['Mejor_Opcion']=='MET Querétaro'])],
            ['⚖️ Neutrales', len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual'])],
            ['💰 Ahorro total (km)', round(df_analisis['Ahorro_km'].sum(),2)],
            ['📈 Ahorro promedio (km)', round(df_analisis['Ahorro_km'].mean(),2)],
            ['🏆 Máximo ahorro individual (km)', round(df_analisis['Ahorro_km'].max(),2)],
            ['🔢 Mediana ahorro (km)', round(df_analisis['Ahorro_km'].median(),2)],
            ['📊 % Celaya', round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya'])/len(df_analisis)*100,1)],
            ['📊 % Querétaro', round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Querétaro'])/len(df_analisis)*100,1)],
            ['📊 % Neutral', round(len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual'])/len(df_analisis)*100,1)],
            ['🗓️ Fecha análisis', timestamp],
            ['📄 Archivo fuente', os.path.basename(directorio)],
        ]
        df_resumen = pd.DataFrame(resumen, columns=['Métrica','Valor'])
        df_resumen.to_excel(writer, sheet_name='Resumen_Ejecutivo', index=False)
        df_analisis.to_excel(writer, sheet_name='Analisis_Detallado', index=False)
        resumen_ruta = df_analisis.groupby('Ruta').agg({
            'CodCli':'count',
            'Ahorro_km':'sum',
            'Mejor_Opcion':lambda x: x.value_counts().to_dict()
        }).rename(columns={'CodCli':'Total_Clientes','Ahorro_km':'Ahorro_Total'})
        resumen_ruta.to_excel(writer, sheet_name='Resumen_por_Ruta')
    wb = openpyxl.load_workbook(archivo_excel)
    ws = wb['Resumen_Ejecutivo']
    img = XLImage(grafica_buf)
    img.width = 420
    img.height = 220
    img.anchor = 'C2'
    ws.add_image(img)
    naranja_oscuro = 'C25A00'
    for row in ws.iter_rows(min_row=2, max_row=2, min_col=1, max_col=2):
        for cell in row:
            cell.font = Font(bold=True, color='FFFFFF', size=14)
            cell.fill = PatternFill(start_color=naranja_oscuro, end_color=naranja_oscuro, fill_type='solid')
            cell.alignment = Alignment(horizontal='center')
    for row in ws.iter_rows(min_row=3, max_row=ws.max_row, min_col=1, max_col=2):
        for cell in row:
            cell.font = Font(size=12)
            cell.alignment = Alignment(horizontal='left')
            if row[0].value and 'Celaya' in str(row[0].value):
                cell.fill = PatternFill(start_color='FFCCCC', end_color='FFCCCC', fill_type='solid')
            elif row[0].value and 'Querétaro' in str(row[0].value):
                cell.fill = PatternFill(start_color='CCE5FF', end_color='CCE5FF', fill_type='solid')
            elif row[0].value and 'Neutral' in str(row[0].value):
                cell.fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
            elif row[0].value and 'Ahorro' in str(row[0].value):
                cell.fill = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')
    ws.column_dimensions['A'].width = 32
    ws.column_dimensions['B'].width = 22
    wb.save(archivo_excel)
    return archivo_excel

def crear_mapa_cluster(df_analisis, gdf_cdr, directorio, timestamp):
    centro_lat = df_analisis['U_latitud'].mean()
    centro_lon = df_analisis['U_longitud'].mean()
    mapa = folium.Map(location=[centro_lat, centro_lon], zoom_start=8, tiles='OpenStreetMap')
    rutas = df_analisis['Ruta'].unique() if 'Ruta' in df_analisis.columns else ['General']
    colores_rutas = sns.color_palette('husl', len(rutas)).as_hex()
    for idx, ruta in enumerate(rutas):
        grupo = df_analisis[df_analisis['Ruta'] == ruta] if 'Ruta' in df_analisis.columns else df_analisis
        capa_ruta = folium.FeatureGroup(name=f'Ruta: {ruta}')
        cluster = MarkerCluster(name=f'Clientes {ruta}')
        if len(grupo) > 2:
            from shapely.geometry import MultiPoint
            puntos = [Point(row['U_longitud'], row['U_latitud']) for _, row in grupo.iterrows()]
            multipoint = MultiPoint(puntos)
            hull = multipoint.convex_hull
            if hull.geom_type == 'Polygon':
                coords = [(lat, lon) for lon, lat in hull.exterior.coords]
                popup_ruta = f'''<div style="width:210px; padding:9px 10px; background:{colores_rutas[idx]}; color:#fff; border-radius:7px; box-shadow:0 1px 6px #aaa; font-size:9px; font-weight:bold; text-align:center;"><span style='font-size:11px;'>🛣️ Ruta: {ruta}</span></div>'''
                folium.Polygon(locations=coords, color=colores_rutas[idx], fill=True, fill_opacity=0.25, popup=folium.Popup(popup_ruta, max_width=225)).add_to(capa_ruta)
        for _, row in grupo.iterrows():
            popup_html = f'''<div style="width:420px; padding:16px 18px; background:#fff; border-radius:14px; box-shadow:0 2px 12px #aaa; font-size:17px;"><div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;"><span style="font-weight:bold; color:{colores_rutas[idx]}; font-size:20px;">Cliente: {row.get('CodCli','')}</span><span style="font-size:28px;">🧑‍💼</span></div><div style="display:flex; justify-content:space-between; margin-bottom:8px;"><span>📚 <b>Nombre:</b> {row.get('Nombre','')}</span><span>🛣️ <b>Ruta:</b> {row.get('Ruta','')}</span></div><div style="display:flex; justify-content:space-between; margin-bottom:8px;"><span>📍 <b>Celaya:</b> {row['Dist_Celaya_km']} km</span><span>📍 <b>Querétaro:</b> {row['Dist_Querétaro_km']} km</span></div><div style="display:flex; justify-content:space-between; margin-bottom:8px;"><span>🎯 <b>Mejor:</b> {row['Mejor_Opcion']}</span><span>💡 <b>Ahorro:</b> {row['Ahorro_km']} km</span></div></div>'''
            folium.Marker(location=[row['U_latitud'], row['U_longitud']], popup=folium.Popup(popup_html, max_width=450), icon=folium.Icon(color='blue', icon='user', prefix='fa')).add_to(cluster)
        cluster.add_to(capa_ruta)
        capa_ruta.add_to(mapa)
    icon_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
    cdr_icon = folium.CustomIcon(icon_image=icon_path, icon_size=(38,38))
    for _, cdr in gdf_cdr.iterrows():
        meet_color = '#FF6B00' if cdr['CodMET']=='MET Celaya' else '#00529B'
        meet_icon = '🏢'
        popup_html = f'''<div style="width:320px; padding:14px; background:{meet_color}; color:#fff; border-radius:12px; box-shadow:0 2px 8px #aaa; text-align:center; font-size:18px;"><div style="font-size:36px;">{meet_icon}</div><div style="font-weight:bold; font-size:22px; margin-top:6px;">{cdr['CodMET']}</div><div style="font-size:16px; margin-top:4px;">Lat: {cdr['U_latitud']}<br>Lon: {cdr['U_longitud']}</div></div>'''
        folium.Marker(location=[cdr['U_latitud'], cdr['U_longitud']], popup=folium.Popup(popup_html, max_width=350), icon=cdr_icon).add_to(mapa)
    folium.LayerControl(collapsed=False, position='topleft').add_to(mapa)
    titulo_html = '''<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background-color: white; padding: 7px 20px; border: 2px solid #333; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);"><h2 style="margin: 0; color: #333; text-align: center; font-size:14px;"><span style='font-size:14px;'>🗺️ MAPA DE CONVENIENCIA - ANÁLISIS MET</span></h2><p style="margin: 4px 0 0 0; text-align: center; color: #666; font-size: 9px;">Celaya vs Querétaro | {} Clientes Analizados</p></div>'''.format(len(df_analisis))
    mapa.get_root().html.add_child(folium.Element(titulo_html))
    resumen_html = '''<div style="position: fixed; top: 130px; right: 35px; width: 187px; background-color: white; border: 2px solid #333; z-index: 9997; font-size: 8.8px; padding: 10px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);"><h3 style="margin-top: 0; color: #333; text-align: left; border-bottom: 2px solid #ddd; padding-bottom: 5px; font-size:11.4px;"><span style='font-size:11.4px;'>📊 ANÁLISIS MET - RESULTADOS</span></h3><p style="margin: 5px 0; font-weight: bold; color: #333; font-size:9.9px;"><span style='font-size:9.9px;'>🧮 Total clientes: {}</span></p><h4 style="margin: 6px 0 4px 0; color: #444; font-size:9.4px;">🎯 DISTRIBUCIÓN POR MET:</h4><p style="margin: 3px 0;"><span style="color: #FF0000; font-size: 9.4px;">●</span> <strong>MET Celaya:</strong> {} clientes ({}%)</p><p style="margin: 3px 0;"><span style="color: #0000FF; font-size: 9.4px;">●</span> <strong>MET Querétaro:</strong> {} clientes ({}%)</p><p style="margin: 3px 0;"><span style="color: #FFA500; font-size: 9.4px;">●</span> <strong>Neutral:</strong> {} clientes ({}%)</p></div>'''.format(len(df_analisis),len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya']), round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya'])/len(df_analisis)*100,1),len(df_analisis[df_analisis['Mejor_Opcion']=='MET Querétaro']), round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Querétaro'])/len(df_analisis)*100,1),len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual']), round(len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual'])/len(df_analisis)*100,1))
    mapa.get_root().html.add_child(folium.Element(resumen_html))
    archivo_mapa = os.path.join(directorio, f'Mapa_Cluster_{timestamp}.html')
    mapa.save(archivo_mapa)
    return archivo_mapa

# ================== FLUJO PRINCIPAL ==================
ARCHIVO_DATOS = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Clientes_Ruta.xlsx'
DIRECTORIO_RESULTADOS = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Resultados'
CANTIDAD_CLIENTES_ANALIZAR = 4415
TOLERANCIA_NEUTRAL_KM = 2
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs(DIRECTORIO_RESULTADOS, exist_ok=True)

gdf_clientes, gdf_cdr = cargar_datos(ARCHIVO_DATOS)
if gdf_clientes is None or gdf_cdr is None:
    raise Exception('No se pudieron cargar los datos')

# Usar servidor público OSRM (no requiere local_server_url)
df_analisis = analizar_distancias_osrm(gdf_clientes, gdf_cdr, cantidad=CANTIDAD_CLIENTES_ANALIZAR, tolerancia=TOLERANCIA_NEUTRAL_KM)
archivo_mapa = crear_mapa_cluster(df_analisis, gdf_cdr, DIRECTORIO_RESULTADOS, timestamp)
grafica_buf = crear_visualizaciones(df_analisis)
archivo_excel = exportar_excel(df_analisis, DIRECTORIO_RESULTADOS, timestamp, CANTIDAD_CLIENTES_ANALIZAR, grafica_buf)
from IPython.display import display, IFrame
display(IFrame(src=archivo_mapa, width=900, height=600))
print('Reporte Excel generado en:', archivo_excel)

Reporte Excel generado en: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Resultados\Analisis_CDR_Adaptable_4410_Clientes_20250822_125704.xlsx


In [ ]:
# ================== VISUALIZACIONES ==================
def crear_visualizaciones(df_analisis):
    buf = BytesIO()
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    colores = ['#FF6B00', '#00529B', '#FFA500']
    asignacion = df_analisis['Mejor_Opcion'].value_counts()
    axs[0].pie(asignacion.values, labels=asignacion.index, autopct='%1.1f%%', colors=colores[:len(asignacion)], startangle=90)
    axs[0].set_title('Distribución de Asignación de Clientes', fontsize=13, color='#FF6B00')
    axs[1].hist(df_analisis['Dist_Celaya_km'], bins=25, alpha=0.7, label='MET Celaya', color='#FF6B00', edgecolor='black')
    axs[1].hist(df_analisis['Dist_Querétaro_km'], bins=25, alpha=0.7, label='MET Querétaro', color='#00529B', edgecolor='black')
    axs[1].set_xlabel('Distancia (km)', fontsize=11)
    axs[1].set_ylabel('Frecuencia', fontsize=11)
    axs[1].set_title('Histograma de Distancias por MEET', fontsize=13, color='#FF6B00')
    axs[1].legend(fontsize=11)
    axs[1].grid(True, alpha=0.3)
    colores_disp = {'MET Celaya': '#FF6B00', 'MET Querétaro': '#00529B', 'Se queda igual': '#FFA500'}
    for meet, color in colores_disp.items():
        subset = df_analisis[df_analisis['Mejor_Opcion'] == meet]
        axs[2].scatter(subset['U_longitud'], subset['U_latitud'], s=20, alpha=0.7, c=color, label=meet)
    axs[2].set_xlabel('Longitud', fontsize=11)
    axs[2].set_ylabel('Latitud', fontsize=11)
    axs[2].set_title('Dispersión de Clientes por MEET', fontsize=13, color='#FF6B00')
    axs[2].legend(fontsize=11)
    plt.tight_layout()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf

# ================== EXPORTACIÓN A EXCEL ==================
def exportar_excel(df_analisis, directorio, timestamp, cantidad, grafica_buf):
    archivo_excel = os.path.join(directorio, f'Analisis_CDR_Adaptable_{len(df_analisis)}_Clientes_{timestamp}.xlsx')
    with pd.ExcelWriter(archivo_excel, engine='openpyxl') as writer:
        resumen = [
            ['📊 MÉTRICAS PRINCIPALES', ''],
            ['👥 Total clientes analizados', len(df_analisis)],
            ['🏢 MET Celaya', len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya'])],
            ['🏢 MET Querétaro', len(df_analisis[df_analisis['Mejor_Opcion']=='MET Querétaro'])],
            ['⚖️ Neutrales', len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual'])],
            ['💰 Ahorro total (km)', round(df_analisis['Ahorro_km'].sum(),2)],
            ['📈 Ahorro promedio (km)', round(df_analisis['Ahorro_km'].mean(),2)],
            ['🏆 Máximo ahorro individual (km)', round(df_analisis['Ahorro_km'].max(),2)],
            ['🔢 Mediana ahorro (km)', round(df_analisis['Ahorro_km'].median(),2)],
            ['📊 % Celaya', round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya'])/len(df_analisis)*100,1)],
            ['📊 % Querétaro', round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Querétaro'])/len(df_analisis)*100,1)],
            ['📊 % Neutral', round(len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual'])/len(df_analisis)*100,1)],
            ['🗓️ Fecha análisis', timestamp],
            ['📄 Archivo fuente', os.path.basename(directorio)],
        ]
        df_resumen = pd.DataFrame(resumen, columns=['Métrica','Valor'])
        df_resumen.to_excel(writer, sheet_name='Resumen_Ejecutivo', index=False)
        df_analisis.to_excel(writer, sheet_name='Analisis_Detallado', index=False)
        resumen_ruta = df_analisis.groupby('Ruta').agg({
            'CodCli':'count',
            'Ahorro_km':'sum',
            'Mejor_Opcion':lambda x: x.value_counts().to_dict()
        }).rename(columns={'CodCli':'Total_Clientes','Ahorro_km':'Ahorro_Total'})
        resumen_ruta.to_excel(writer, sheet_name='Resumen_por_Ruta')
    wb = openpyxl.load_workbook(archivo_excel)
    ws = wb['Resumen_Ejecutivo']
    img = XLImage(grafica_buf)
    img.width = 420
    img.height = 220
    img.anchor = 'C2'
    ws.add_image(img)
    naranja_oscuro = 'C25A00'
    for row in ws.iter_rows(min_row=2, max_row=2, min_col=1, max_col=2):
        for cell in row:
            cell.font = Font(bold=True, color='FFFFFF', size=14)
            cell.fill = PatternFill(start_color=naranja_oscuro, end_color=naranja_oscuro, fill_type='solid')
            cell.alignment = Alignment(horizontal='center')
    for row in ws.iter_rows(min_row=3, max_row=ws.max_row, min_col=1, max_col=2):
        for cell in row:
            cell.font = Font(size=12)
            cell.alignment = Alignment(horizontal='left')
            if row[0].value and 'Celaya' in str(row[0].value):
                cell.fill = PatternFill(start_color='FFCCCC', end_color='FFCCCC', fill_type='solid')
            elif row[0].value and 'Querétaro' in str(row[0].value):
                cell.fill = PatternFill(start_color='CCE5FF', end_color='CCE5FF', fill_type='solid')
            elif row[0].value and 'Neutral' in str(row[0].value):
                cell.fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
            elif row[0].value and 'Ahorro' in str(row[0].value):
                cell.fill = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')
    ws.column_dimensions['A'].width = 32
    ws.column_dimensions['B'].width = 22
    wb.save(archivo_excel)
    return archivo_excel

# ================== CREACIÓN DE MAPA CON POLÍGONOS ENVOLVENTES POR RUTA ==================
def crear_mapa_cluster(df_analisis, gdf_cdr, directorio, timestamp):
    centro_lat = df_analisis['U_latitud'].mean()
    centro_lon = df_analisis['U_longitud'].mean()
    mapa = folium.Map(location=[centro_lat, centro_lon], zoom_start=8, tiles='OpenStreetMap')
    rutas = df_analisis['Ruta'].unique() if 'Ruta' in df_analisis.columns else ['General']
    colores_rutas = sns.color_palette('husl', len(rutas)).as_hex()
    for idx, ruta in enumerate(rutas):
        grupo = df_analisis[df_analisis['Ruta'] == ruta] if 'Ruta' in df_analisis.columns else df_analisis
        capa_ruta = folium.FeatureGroup(name=f'Ruta: {ruta}')
        cluster = MarkerCluster(name=f'Clientes {ruta}')
        if len(grupo) > 2:
            from shapely.geometry import MultiPoint
            puntos = [Point(row['U_longitud'], row['U_latitud']) for _, row in grupo.iterrows()]
            multipoint = MultiPoint(puntos)
            hull = multipoint.convex_hull
            if hull.geom_type == 'Polygon':
                coords = [(lat, lon) for lon, lat in hull.exterior.coords]
                popup_ruta = f'''<div style="width:210px; padding:9px 10px; background:{colores_rutas[idx]}; color:#fff; border-radius:7px; box-shadow:0 1px 6px #aaa; font-size:9px; font-weight:bold; text-align:center;"><span style='font-size:11px;'>🛣️ Ruta: {ruta}</span></div>'''
                folium.Polygon(locations=coords, color=colores_rutas[idx], fill=True, fill_opacity=0.25, popup=folium.Popup(popup_ruta, max_width=225)).add_to(capa_ruta)
        for _, row in grupo.iterrows():
            popup_html = f'''<div style="width:420px; padding:16px 18px; background:#fff; border-radius:14px; box-shadow:0 2px 12px #aaa; font-size:17px;"><div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;"><span style="font-weight:bold; color:{colores_rutas[idx]}; font-size:20px;">Cliente: {row.get('CodCli','')}</span><span style="font-size:28px;">🧑‍💼</span></div><div style="display:flex; justify-content:space-between; margin-bottom:8px;"><span>📚 <b>Nombre:</b> {row.get('Nombre','')}</span><span>🛣️ <b>Ruta:</b> {row.get('Ruta','')}</span></div><div style="display:flex; justify-content:space-between; margin-bottom:8px;"><span>📍 <b>Celaya:</b> {row['Dist_Celaya_km']} km</span><span>📍 <b>Querétaro:</b> {row['Dist_Querétaro_km']} km</span></div><div style="display:flex; justify-content:space-between; margin-bottom:8px;"><span>🎯 <b>Mejor:</b> {row['Mejor_Opcion']}</span><span>💡 <b>Ahorro:</b> {row['Ahorro_km']} km</span></div></div>'''
            folium.Marker(location=[row['U_latitud'], row['U_longitud']], popup=folium.Popup(popup_html, max_width=450), icon=folium.Icon(color='blue', icon='user', prefix='fa')).add_to(cluster)
        cluster.add_to(capa_ruta)
        capa_ruta.add_to(mapa)
    icon_path = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\95_24.png'
    cdr_icon = folium.CustomIcon(icon_image=icon_path, icon_size=(38,38))
    for _, cdr in gdf_cdr.iterrows():
        meet_color = '#FF6B00' if cdr['CodMET']=='MET Celaya' else '#00529B'
        meet_icon = '🏢'
        popup_html = f'''<div style="width:320px; padding:14px; background:{meet_color}; color:#fff; border-radius:12px; box-shadow:0 2px 8px #aaa; text-align:center; font-size:18px;"><div style="font-size:36px;">{meet_icon}</div><div style="font-weight:bold; font-size:22px; margin-top:6px;">{cdr['CodMET']}</div><div style="font-size:16px; margin-top:4px;">Lat: {cdr['U_latitud']}<br>Lon: {cdr['U_longitud']}</div></div>'''
        folium.Marker(location=[cdr['U_latitud'], cdr['U_longitud']], popup=folium.Popup(popup_html, max_width=350), icon=cdr_icon).add_to(mapa)
    folium.LayerControl(collapsed=False, position='topleft').add_to(mapa)
    titulo_html = '''<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 9998; background-color: white; padding: 7px 20px; border: 2px solid #333; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);"><h2 style="margin: 0; color: #333; text-align: center; font-size:14px;"><span style='font-size:14px;'>🗺️ MAPA DE CONVENIENCIA - ANÁLISIS MET</span></h2><p style="margin: 4px 0 0 0; text-align: center; color: #666; font-size: 9px;">Celaya vs Querétaro | {} Clientes Analizados</p></div>'''.format(len(df_analisis))
    mapa.get_root().html.add_child(folium.Element(titulo_html))
    resumen_html = '''<div style="position: fixed; top: 130px; right: 35px; width: 187px; background-color: white; border: 2px solid #333; z-index: 9997; font-size: 8.8px; padding: 10px; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);"><h3 style="margin-top: 0; color: #333; text-align: left; border-bottom: 2px solid #ddd; padding-bottom: 5px; font-size:11.4px;"><span style='font-size:11.4px;'>📊 ANÁLISIS MET - RESULTADOS</span></h3><p style="margin: 5px 0; font-weight: bold; color: #333; font-size:9.9px;"><span style='font-size:9.9px;'>🧮 Total clientes: {}</span></p><h4 style="margin: 6px 0 4px 0; color: #444; font-size:9.4px;">🎯 DISTRIBUCIÓN POR MET:</h4><p style="margin: 3px 0;"><span style="color: #FF0000; font-size: 9.4px;">●</span> <strong>MET Celaya:</strong> {} clientes ({}%)</p><p style="margin: 3px 0;"><span style="color: #0000FF; font-size: 9.4px;">●</span> <strong>MET Querétaro:</strong> {} clientes ({}%)</p><p style="margin: 3px 0;"><span style="color: #FFA500; font-size: 9.4px;">●</span> <strong>Neutral:</strong> {} clientes ({}%)</p></div>'''.format(len(df_analisis),len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya']), round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Celaya'])/len(df_analisis)*100,1),len(df_analisis[df_analisis['Mejor_Opcion']=='MET Querétaro']), round(len(df_analisis[df_analisis['Mejor_Opcion']=='MET Querétaro'])/len(df_analisis)*100,1),len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual']), round(len(df_analisis[df_analisis['Mejor_Opcion']=='Se queda igual'])/len(df_analisis)*100,1))
    mapa.get_root().html.add_child(folium.Element(resumen_html))
    archivo_mapa = os.path.join(directorio, f'Mapa_Cluster_{timestamp}.html')
    mapa.save(archivo_mapa)
    return archivo_mapa

# ================== FLUJO PRINCIPAL ==================
ARCHIVO_DATOS = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Clientes_Ruta.xlsx'
DIRECTORIO_RESULTADOS = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Resultados'
CANTIDAD_CLIENTES_ANALIZAR = 4415
TOLERANCIA_NEUTRAL_KM = 2
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs(DIRECTORIO_RESULTADOS, exist_ok=True)

gdf_clientes, gdf_cdr = cargar_datos(ARCHIVO_DATOS)
if gdf_clientes is None or gdf_cdr is None:
    raise Exception('No se pudieron cargar los datos')

# Usar servidor público OSRM (no requiere local_server_url)
df_analisis = analizar_distancias_osrm(gdf_clientes, gdf_cdr, cantidad=CANTIDAD_CLIENTES_ANALIZAR, tolerancia=TOLERANCIA_NEUTRAL_KM)
archivo_mapa = crear_mapa_cluster(df_analisis, gdf_cdr, DIRECTORIO_RESULTADOS, timestamp)
grafica_buf = crear_visualizaciones(df_analisis)
archivo_excel = exportar_excel(df_analisis, DIRECTORIO_RESULTADOS, timestamp, CANTIDAD_CLIENTES_ANALIZAR, grafica_buf)
from IPython.display import display, IFrame
display(IFrame(src=archivo_mapa, width=900, height=600))
print('Reporte Excel generado en:', archivo_excel)